# Application on Dataset - ConceptNet Context

In [ ]:
!pip install -q -U google-generativeai pandas scikit-learn tqdm

In [ ]:
# ===============================================================================
# STEP 1: INSTALLATION, IMPORTS AND CONFIGURATION
# ===============================================================================
import pandas as pd
import google.generativeai as genai
from google.colab import userdata
from sklearn.metrics import accuracy_score, f1_score
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
import re
from google.colab import drive

In [ ]:
# Gemini API Key Configuration
try:
    # Use Colab secret (recommended)
    GOOGLE_API_KEY = '' # Your API key here
    genai.configure(api_key=GOOGLE_API_KEY)
except Exception as e:
    print(f"'GEMINI_API_KEY' not found or invalid. Set it in Colab Secrets. Error: {e}")

In [ ]:
# ===============================================================================
# STEP 2: DATA LOADING
# ===============================================================================
drive.mount('/content/drive')

try:
    df = pd.read_csv('/content/drive/MyDrive/ProjectFolder/prompts_IBM.csv') # for experiments 1-3
    df = pd.read_csv('/content/drive/MyDrive/ProjectFolder/prompts_IBM_evaluated_3_ways.csv') # for experiments 3-6
    print(f"File 'prompts_IBM.csv' loaded successfully. Total of {len(df)} rows.")
except FileNotFoundError:
    print("ERROR: File 'prompts_IBM.csv' not found. Make sure it was generated and is in the environment.")
    df = None

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
# ==============================================================================
# ETAPA 3: FUNÇÕES DE PROCESSAMENTO E CHAMADA À API (VERSÃO CORRIGIDA)
# ==============================================================================

def _call_gemini_api_with_retry(prompt, model):
    """
    Função auxiliar que chama a API com lógica de nova tentativa.
    O prompt já deve conter a instrução completa.
    """
    for attempt in range(2): # Tenta até 2 vezes
        try:
            response = model.generate_content(prompt)
            answer = response.text.strip().upper()
            if "ATTACK" in answer:
                return "ATTACK"
            elif "SUPPORT" in answer:
                return "SUPPORT"
            else:
                return "N/A" # Resposta inesperada
        except Exception as e:
            if attempt == 1: # Se for a última tentativa
                print(f"Erro final ao processar prompt: {str(e)}")
                return "ERROR"
    return "ERROR"

## Experiments 1-3

In [ ]:
# ==============================================================================
# FUNÇÃO DE PROCESSAMENTO ATUALIZADA (dependência para Etapas 4 e 5)
# ==============================================================================

# Certifique-se de que a função _call_gemini_api_with_retry está definida na célula anterior.

def process_row(row, model):
    """
    Processa uma única linha, obtendo respostas para os três cenários:
    1. Sem Contexto
    2. Com Contexto KG
    3. Com Contexto KG + Sentimento
    """
    # Define os três prompts
    prompt_with_sentiment = row['prompt_with_context_and_sentiment']
    prompt_with_kg_only = row['final_prompt']

    context_pattern = r"Knowledge Graph Context:.*?Question:"
    prompt_without_context = re.sub(context_pattern, "Question:", prompt_with_kg_only, flags=re.DOTALL)

    # Chama a API para os três cenários
    answer_with_sentiment = _call_gemini_api_with_retry(prompt_with_sentiment, model)
    answer_with_kg_only = _call_gemini_api_with_retry(prompt_with_kg_only, model)
    answer_without_context = _call_gemini_api_with_retry(prompt_without_context, model)

    # Retorna os 3 resultados e o prompt gerado para a célula de teste
    return answer_without_context, answer_with_kg_only, answer_with_sentiment, prompt_without_context

In [ ]:
# ==============================================================================
# NOVA ETAPA: PRÉ-PROCESSAMENTO DE ANÁLISE DE SENTIMENTO (COM RETRY)
# ==============================================================================
import time
import random
from tqdm.auto import tqdm

def get_sentiment(text, model):
    """
    Chama o modelo Gemini para classificar o sentimento de um texto.
    Retorna 'POSITIVE', 'NEGATIVE', ou 'NEUTRAL'.
    --- INCLUI LÓGICA DE RETRY COM EXPONENTIAL BACKOFF ---
    """
    prompt = f"Analyze the sentiment of the following text. Respond ONLY with the word POSITIVE, NEGATIVE, or NEUTRAL.\n\nText: \"{text}\"\n\nSentiment:"

    # Lógica de retry: Tenta até 3 vezes com um tempo de espera crescente
    for attempt in range(3):
        try:
            response = model.generate_content(prompt)
            answer = response.text.strip().upper()
            if answer in ["POSITIVE", "NEGATIVE", "NEUTRAL"]:
                return answer
            else:
                return "NEUTRAL" # Retorna um padrão seguro se a resposta for válida, mas inesperada
        except Exception as e:
            # Se for um erro de recurso indisponível (como 503), espera e tenta de novo
            if "resource_exhausted" in str(e) or "503" in str(e):
                # Espera 2^n segundos + um valor aleatório (jitter) para evitar picos de requisição
                wait_time = (2 ** attempt) + random.random()
                print(f"Rate limit atingido. Tentativa {attempt + 1}/3. Esperando {wait_time:.2f} segundos...")
                time.sleep(wait_time)
            else:
                # Se for outro tipo de erro, registra e retorna ERROR
                print(f"Erro não relacionado a rate limit: '{text[:50]}...'. Erro: {e}")
                return "ERROR"

    # Se todas as tentativas falharem
    print(f"Todas as 3 tentativas falharam para o texto: '{text[:50]}...'.")
    return "ERROR"

# Verifique se o DataFrame df foi carregado e não está vazio
if 'df' in locals() and not df.empty:
    print("Iniciando a análise de sentimento para o DataFrame completo...")

    if 'model' not in locals():
        model = genai.GenerativeModel('gemini-1.5-flash')

    # --- Aplica a análise de sentimento com barra de progresso em todo o DataFrame ---
    tqdm.pandas(desc="Analisando sentimento dos argumentos")
    df['argument_sentiment'] = df['argument'].progress_apply(lambda x: get_sentiment(x, model))

    tqdm.pandas(desc="Analisando sentimento dos tópicos")
    df['topic_sentiment'] = df['topic'].progress_apply(lambda x: get_sentiment(x, model))

    # --- Constrói o bloco de texto de sentimento ---
    df['sentiment_block'] = df.apply(
        lambda row: (
            "Sentiment Analysis:\n"
            f"Argument has a {row['argument_sentiment']} sentiment\n"
            f"Topic has a {row['topic_sentiment']} sentiment"
        ),
        axis=1
    )

    # --- Cria a nova coluna de prompt enriquecido ---
    new_prompt_template = (
        "Your task is to analyze an argument and a belief, using the provided 'Knowledge Graph Context' which shows explanations of concepts in the argument and belief, and using the sentiment of both argument and belief. "
        "Based on this, decide if the argument attacks or supports the belief. Your answer must be ONLY the word 'ATTACK' or 'SUPPORT'."
    )

    df['prompt_with_context_and_sentiment'] = df.apply(
        lambda row: (
            f"{new_prompt_template}\n\n"
            f"{row['sentiment_block']}\n\n"
            f"Knowledge Graph Context: {row['knowledge_context']}\n\n"
            f"Question: {row['final_prompt'].split('Question:')[1].strip()}"
        ),
        axis=1
    )

    print("\nPré-processamento de sentimento concluído!")
    print("Novas colunas foram criadas no DataFrame 'df'.")

    # Exibe um exemplo do novo prompt a partir do DataFrame principal
    print("\n--- Exemplo de Novo Prompt Enriquecido ---")
    print(df['prompt_with_context_and_sentiment'].iloc[0])

else:
    print("DataFrame 'df' não encontrado ou vazio. Execute as células de carregamento de dados primeiro.")

In [ ]:
df.head()

In [ ]:
print(df['prompt_with_context_and_sentiment'][0])

In [ ]:
df.to_csv('/content/drive/MyDrive/.../prompts_IBM_with_context_and_sentiment.csv', index=False, encoding='utf-8')

In [ ]:
# ==============================================================================
# ETAPA 4: TESTE EM UM PROMPT ALEATÓRIO (VERSÃO PARA 3 CENÁRIOS)
# ==============================================================================

if df is not None:
    print("\n--- Iniciando teste em um prompt aleatório para 3 cenários ---")

    if 'model' not in locals():
        model = genai.GenerativeModel('gemini-1.5-flash')

    random_row = df.sample(1).iloc[0]

    # Processa a linha usando a nova função
    res_without, res_with, res_with_sent, gen_prompt_without = process_row(random_row, model)

    print("\n[PROMPT 1: SEM CONTEXTO]")
    print(gen_prompt_without)
    print("-" * 20)

    print("\n[PROMPT 2: COM CONTEXTO KG]")
    print(random_row['final_prompt'])
    print("-" * 20)

    print("\n[PROMPT 3: COM CONTEXTO KG + SENTIMENTO]")
    print(random_row['prompt_with_context_and_sentiment'])
    print("-" * 20)

    print("\n[RESULTADOS DO TESTE]")
    print(f"Resposta Correta (Ground Truth): {random_row['ground_truth_stance']}")
    print(f"Resposta do Gemini (Sem Contexto):                {res_without}")
    print(f"Resposta do Gemini (Com Contexto KG):             {res_with}")
    print(f"Resposta do Gemini (Com Contexto KG + Sentimento): {res_with_sent}")
    print("\n--- Teste concluído ---")

In [ ]:
# ==============================================================================
# ETAPA 5: EXECUÇÃO PARALELA (VERSÃO PARA 3 CENÁRIOS)
# ==============================================================================

if df is not None:
    print("\n--- Iniciando processamento em lote para 3 cenários ---")

    if 'model' not in locals():
        model = genai.GenerativeModel('gemini-1.5-flash')

    results = [None] * len(df)

    with ThreadPoolExecutor(max_workers=10) as executor:
        futures = {executor.submit(process_row, row, model): index for index, row in df.iterrows()}

        for future in tqdm(as_completed(futures), total=len(df), desc="Avaliando 3 cenários"):
            original_index = futures[future]
            try:
                # Desempacota os 4 valores e ignora o último
                res_without, res_with, res_with_sent, _ = future.result()
                results[original_index] = (res_without, res_with, res_with_sent)

            except Exception as e:
                print(f"Erro na thread para o índice {original_index}: {e}")
                results[original_index] = ("ERROR", "ERROR", "ERROR")

    # Atribui os resultados às três colunas do DataFrame
    df['gemini_without_context'] = [res[0] if res else "ERROR" for res in results]
    df['gemini_with_context'] = [res[1] if res else "ERROR" for res in results]
    df['gemini_with_context_sentiment'] = [res[2] if res else "ERROR" for res in results]

    # Salva o DataFrame final com os resultados
    output_filename = '/content/drive/MyDrive/.../prompts_IBM_evaluated_3_ways.csv'
    df.to_csv(output_filename, index=False, encoding='utf-8')
    print(f"\n--- Processamento concluído. Resultados salvos em '{output_filename}' ---")

In [ ]:
# ==============================================================================
# ETAPA 6: ANÁLISE E CÁLCULO DE MÉTRICAS (VERSÃO PARA 3 CENÁRIOS)
# ==============================================================================

def analyze_results_3_ways(results_df):
    """
    Calcula métricas e analisa os 3 cenários de prompt.
    """
    print("\n--- ANÁLISE DE DESEMPENHO ---")

    y_true = results_df['ground_truth_stance']
    scenarios = {
        "Sem Contexto": "gemini_without_context",
        "Com Contexto KG": "gemini_with_context",
        "Com Contexto KG + Sentimento": "gemini_with_context_sentiment"
    }

    for name, col in scenarios.items():
        y_pred = results_df[col]
        valid_results = results_df[results_df[col].isin(['SUPPORT', 'ATTACK'])]

        acc = accuracy_score(y_true, y_pred)
        f1 = f1_score(valid_results['ground_truth_stance'], valid_results[col], average='weighted')

        print(f"\n[Resultados para: {name}]")
        print(f"Acurácia: {acc:.4f} (incluindo erros)")
        print(f"F1-Score (Weighted): {f1:.4f} (apenas sobre respostas válidas)")

    # Análises comparativas
    y_pred_without = results_df['gemini_without_context']
    y_pred_with = results_df['gemini_with_context']
    y_pred_with_sent = results_df['gemini_with_context_sentiment']

    context_made_worse = results_df[(y_pred_without == y_true) & (y_pred_with != y_true)]
    context_made_better = results_df[(y_pred_without != y_true) & (y_pred_with == y_true)]
    sentiment_made_worse = results_df[(y_pred_with == y_true) & (y_pred_with_sent != y_true)]
    sentiment_made_better = results_df[(y_pred_with != y_true) & (y_pred_with_sent == y_true)]

    print("\n--- ANÁLISE DE CASOS ESPECÍFICOS ---")

    if not context_made_worse.empty:
        print(f"\n[ENCONTRADOS {len(context_made_worse)} CASOS ONDE CONTEXTO KG PIOROU A RESPOSTA (vs Sem Contexto)]")
        # (código para imprimir os detalhes pode ser adicionado aqui se desejado)

    if not context_made_better.empty:
        print(f"\n[ENCONTRADOS {len(context_made_better)} CASOS ONDE CONTEXTO KG MELHOROU A RESPOSTA (vs Sem Contexto)]")

    if not sentiment_made_worse.empty:
        print(f"\n[ENCONTRADOS {len(sentiment_made_worse)} CASOS ONDE O SENTIMENTO PIOROU A RESPOSTA (vs Contexto KG)]")
        for index, row in sentiment_made_worse.iterrows():
            print("\n" + "-"*20)
            print(f"Argumento: {row['argument']}")
            print(f"Resposta Correta: {row['ground_truth_stance']}")
            print(f"Resposta COM Contexto KG (Correta):   {row['gemini_with_context']}")
            print(f"Resposta COM Sentimento (Incorreta): {row['gemini_with_context_sentiment']}")
            print("-" * 20)

    if not sentiment_made_better.empty:
        print(f"\n[ENCONTRADOS {len(sentiment_made_better)} CASOS ONDE O SENTIMENTO MELHOROU A RESPOSTA (vs Contexto KG)]")
        for index, row in sentiment_made_better.iterrows():
            print("\n" + "="*20)
            print(f"Argumento: {row['argument']}")
            print(f"Resposta Correta: {row['ground_truth_stance']}")
            print(f"Resposta COM Contexto KG (Incorreta): {row['gemini_with_context']}")
            print(f"Resposta COM Sentimento (Correta):   {row['gemini_with_context_sentiment']}")
            print("=" * 20)

# Executa a análise se o DataFrame foi processado
if 'gemini_with_context_sentiment' in df.columns:
    analyze_results_3_ways(df)
else:
    print("\nAnálise não pode ser executada pois o processamento em lote para 3 cenários não foi concluído.")

## Experiments 3-6

In [ ]:
# ==============================================================================
# NOVA ETAPA 3.5: PRÉ-PROCESSAMENTO DOS PROMPTS AVANÇADOS
# ==============================================================================

# Verifique se o DataFrame df foi processado com sentimento
if 'prompt_with_context_and_sentiment' in df.columns:
    print("Criando os novos prompts para os experimentos avançados...")

    # --- 1. PROMPT COM CHAIN-OF-THOUGHT (COT) ---
    cot_template = (
        "Reason in the following way:\n"
        "1. Extract from the prompt the argument and topic sentiment. If they have the same sentiment, they are consistent. If not, they are contrastive\n"
        "2. The knowlodge graph helps clarifying any of the terms in the topic or argument?\n"
        "3. Is the clarified term relevant in the argument or topic\n"
        "4. With the the understanding provided by topic and argument, the relevant context provided, and given an relation of consistency or contrative, does the argument provide evidence for or against the topic?\n"
        "5. Conclusion:"
    )

    # Adiciona o template de CoT ao prompt que já continha KG e Sentimento
    df['prompt_cot'] = df.apply(
        lambda row: f"{row['prompt_with_context_and_sentiment']}\n\n{cot_template}",
        axis=1
    )

    # --- 2. PROMPT COM EXEMPLOS (FEW-SHOT) ---
    # Formatando os exemplos que você forneceu
    example_1 = (
        "Argument: \"Marketing promotes consumerism and waste.\"\n"
        "Belief: \"This house believes that advertising is harmful.\"\n"
        "Answer: SUPPORT"
    )
    example_2 = (
        "Argument: \"Unity is seen as an essential feature of the nation and the nation-state.\"\n"
        "Belief: \"This house would embrace multiculturalism.\"\n"
        "Answer: ATTACK"
    )
    example_3 = (
        "Argument: \"Children with many siblings receive fewer resources.\"\n"
        "Belief: \"This house supports the one-child policy of the republic of China.\"\n"
        "Answer: SUPPORT"
    )

    few_shot_header = f"Here are some examples:\n\n[EXAMPLE 1]\n{example_1}\n\n[EXAMPLE 2]\n{example_2}\n\n[EXAMPLE 3]\n{example_3}\n\n---\n\nNow, complete the following task:\n"

    # Adiciona os exemplos ao prompt original (sem contexto)
    df['prompt_few_shot'] = df.apply(
        lambda row: (
            f"{few_shot_header}"
            f"Argument: \"{row['argument']}\"\n"
            f"Belief: \"{row['topic']}\"\n"
            f"Answer:"
        ),
        axis=1
    )

    # --- 3. PROMPT COM EXEMPLOS + CHAIN-OF-THOUGHT (FEW-SHOT COT) ---
    cot_example_1 = (
        "Argument: \"Marketing promotes consumerism and waste.\"\n"
        "Belief: \"This house believes that advertising is harmful.\"\n"
        "Reasoning:\n"
        "1. The argument is negative, and the belief is negative. They are consistent.\n"
        "2. The knowledge graph is not needed to understand the terms.\n"
        "3. N/A.\n"
        "4. Since they are consistent, and the argument points out negative effects of marketing (a form of advertising), it provides evidence for the belief that advertising is harmful.\n"
        "5. Conclusion: The argument provides evidence for the belief.\n"
        "Answer: SUPPORT"
    )
    # (Adaptei a sua anotação 'Con' para 'ATTACK' para manter a consistência)
    cot_example_2 = (
        "Argument: \"Unity is seen as an essential feature of the nation and the nation-state.\"\n"
        "Belief: \"This house would embrace multiculturalism.\"\n"
        "Reasoning:\n"
        "1. The argument is positive towards unity, while the belief is positive towards multiculturalism. These can be seen as contrastive concepts in this context.\n"
        "2. The knowledge graph is not needed.\n"
        "3. N/A.\n"
        "4. The argument champions unity (a single national identity) which is contrary to the idea of embracing multiple cultures (multiculturalism). Therefore, the argument provides evidence against the belief.\n"
        "5. Conclusion: The argument provides evidence against the belief.\n"
        "Answer: ATTACK"
    )

    cot_few_shot_header = f"Follow the reasoning process shown in the examples to complete the final task.\n\n[EXAMPLE 1]\n{cot_example_1}\n\n[EXAMPLE 2]\n{cot_example_2}\n\n---\n\n[FINAL TASK]\n"

    # Usa a mesma estrutura do prompt CoT, mas com o cabeçalho de exemplos
    df['prompt_cot_few_shot'] = df.apply(
        lambda row: (
            f"{cot_few_shot_header}"
            f"Argument: \"{row['argument']}\"\n"
            f"Belief: \"{row['topic']}\"\n"
            f"Knowledge Graph Context: {row['knowledge_context']}\n\n"
            f"{cot_template}\n"
            f"Reasoning:\n"
        ),
        axis=1
    )

    print("Novas colunas 'prompt_cot', 'prompt_few_shot', e 'prompt_cot_few_shot' foram criadas.")
    print("\n--- Exemplo de Prompt Few-Shot com Chain-of-Thought ---")
    print(df['prompt_cot_few_shot'].iloc[0])
else:
    print("Execute as células de pré-processamento anteriores primeiro.")

In [ ]:
# ==============================================================================
# FUNÇÕES DE APOIO (NECESSÁRIAS PARA A ETAPA 4)
# ==============================================================================

def _call_gemini_api_with_retry(prompt, model):
    """
    Função auxiliar que chama a API com lógica de nova tentativa.
    O prompt já deve conter a instrução completa.
    """
    # A instrução está implícita no prompt, então apenas enviamos o prompt
    for attempt in range(2): # Tenta até 2 vezes
        try:
            response = model.generate_content(prompt)
            answer = response.text.strip().upper()
            if "ATTACK" in answer:
                return "ATTACK"
            elif "SUPPORT" in answer:
                return "SUPPORT"
            else:
                # Extrai a última palavra se for uma resposta de CoT
                last_word = answer.split()[-1]
                if "ATTACK" in last_word:
                    return "ATTACK"
                elif "SUPPORT" in last_word:
                    return "SUPPORT"
                return "N/A" # Resposta inesperada
        except Exception as e:
            if attempt == 1: # Se for a última tentativa
                print(f"Erro final ao processar prompt: {str(e)}")
                return "ERROR"
    return "ERROR"

def process_row_new_experiments(row, model):
    """
    Processa uma única linha para os três novos cenários experimentais.
    """
    prompt_cot = row['prompt_cot']
    prompt_few_shot = row['prompt_few_shot']
    prompt_cot_few_shot = row['prompt_cot_few_shot']

    # Chama a API para os três novos cenários
    answer_cot = _call_gemini_api_with_retry(prompt_cot, model)
    answer_few_shot = _call_gemini_api_with_retry(prompt_few_shot, model)
    answer_cot_few_shot = _call_gemini_api_with_retry(prompt_cot_few_shot, model)

    return answer_cot, answer_few_shot, answer_cot_few_shot

# ==============================================================================
# ETAPA 4: TESTE EM UM PROMPT ALEATÓRIO (COM VISUALIZAÇÃO DOS PROMPTS)
# ==============================================================================

if 'df' in locals() and df is not None:
    print("\n--- Iniciando teste em um prompt aleatório para os novos experimentos ---")

    if 'model' not in locals():
        model = genai.GenerativeModel('gemini-1.5-flash')

    # Seleciona uma linha aleatória do DataFrame
    random_row = df.sample(1).iloc[0]

    # Processa a linha para obter as três respostas
    res_cot, res_fs, res_cot_fs = process_row_new_experiments(random_row, model)

    # --- EXIBIÇÃO DOS PROMPTS UTILIZADOS ---
    print("\n" + "="*50)
    print("PROMPT 1: CHAIN-OF-THOUGHT")
    print("="*50)
    print(random_row['prompt_cot'])

    print("\n" + "="*50)
    print("PROMPT 2: FEW-SHOT")
    print("="*50)
    print(random_row['prompt_few_shot'])

    print("\n" + "="*50)
    print("PROMPT 3: COT + FEW-SHOT")
    print("="*50)
    print(random_row['prompt_cot_few_shot'])
    print("="*50)

    # --- EXIBIÇÃO DOS RESULTADOS ---
    print("\n[RESULTADOS DO TESTE]")
    print(f"Resposta Correta (Ground Truth): {random_row['ground_truth_stance']}")
    print("-" * 30)
    print(f"Resposta do Gemini (Chain-of-Thought): {res_cot}")
    print(f"Resposta do Gemini (Few-Shot):           {res_fs}")
    print(f"Resposta do Gemini (CoT + Few-Shot):     {res_cot_fs}")
    print("\n--- Teste concluído ---")

else:
    print("DataFrame 'df' não encontrado. Por favor, execute as células de carregamento de dados primeiro.")

In [ ]:
# ==============================================================================
# ETAPA 5: EXECUÇÃO PARALELA (AVALIANDO NOVOS PROMPTS)
# ==============================================================================

if df is not None:
    print("\n--- Iniciando processamento em lote para os novos experimentos ---")

    if 'model' not in locals():
        model = genai.GenerativeModel('gemini-1.5-flash')

    results = [None] * len(df)

    with ThreadPoolExecutor(max_workers=10) as executor:
        futures = {executor.submit(process_row_new_experiments, row, model): index for index, row in df.iterrows()}

        for future in tqdm(as_completed(futures), total=len(df), desc="Avaliando prompts avançados"):
            original_index = futures[future]
            try:
                results[original_index] = future.result()
            except Exception as e:
                print(f"Erro na thread para o índice {original_index}: {e}")
                results[original_index] = ("ERROR", "ERROR", "ERROR")

    # Atribui os resultados às três novas colunas do DataFrame
    df['gemini_cot'] = [res[0] if res else "ERROR" for res in results]
    df['gemini_few_shot'] = [res[1] if res else "ERROR" for res in results]
    df['gemini_cot_few_shot'] = [res[2] if res else "ERROR" for res in results]

    # Salva o DataFrame final com todos os resultados
    output_filename = '/content/drive/MyDrive/.../Mestrado/prompts_IBM_tought_and_shot.csv'
    df.to_csv(output_filename, index=False, encoding='utf-8')
    print(f"\n--- Processamento concluído. Resultados salvos em '{output_filename}' ---")


In [ ]:

# ==============================================================================
# ETAPA 6: ANÁLISE E CÁLCULO DE MÉTRICAS (AVALIANDO NOVOS PROMPTS)
# ==============================================================================

def analyze_advanced_results(results_df):
    """
    Calcula métricas e analisa os 3 novos cenários de prompt.
    """
    print("\n--- ANÁLISE DE DESEMPENHO DOS NOVOS EXPERIMENTOS ---")

    y_true = results_df['ground_truth_stance']

    # Inclui o melhor resultado anterior (sem contexto) como baseline
    scenarios = {
        "Baseline (Sem Contexto)": "gemini_without_context",
        "Chain-of-Thought": "gemini_cot",
        "Few-Shot": "gemini_few_shot",
        "CoT + Few-Shot": "gemini_cot_few_shot"
    }

    for name, col in scenarios.items():
        if col in results_df.columns:
            y_pred = results_df[col]
            valid_results = results_df[results_df[col].isin(['SUPPORT', 'ATTACK'])]

            acc = accuracy_score(y_true, y_pred)
            f1 = f1_score(valid_results['ground_truth_stance'], valid_results[col], average='weighted')

            print(f"\n[Resultados para: {name}]")
            print(f"Acurácia: {acc:.4f} (incluindo erros)")
            print(f"F1-Score (Weighted): {f1:.4f} (apenas sobre respostas válidas)")
        else:
            print(f"\nColuna de resultados '{col}' não encontrada para o cenário '{name}'.")


# Executa a análise se o DataFrame foi processado
if 'gemini_cot' in df.columns:
    analyze_advanced_results(df)
else:
    print("\nAnálise não pode ser executada pois o processamento em lote para os experimentos avançados não foi concluído.")